In [ ]:
%pip install openai

import pandas as pd
import time
import json
import random

from openai import OpenAI
from pydantic import BaseModel
from concurrent.futures import ThreadPoolExecutor


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
data = pd.read_csv('../../data/scidcc/SciDCC.csv')
data.head()

,Date,Link,Title,Summary,Body,Category,Year
0,"April 15, 2021",https://www.sciencedaily.com/releases/2021/04/...,Impacts of coronavirus lockdowns: New study co...,One consequence of the coronavirus pandemic ha...,The meta-analysis was coordinated by Prof. Ast...,Ozone Holes,2021
1,"April 5, 2021",https://www.sciencedaily.com/releases/2021/04/...,"Ozone pollution harms maize crops, study finds",Although stratospheric ozone protects us by fi...,"Ozone is formed when nitrous oxide, released f...",Ozone Holes,2021
2,"December 2, 2020",https://www.sciencedaily.com/releases/2020/12/...,Ozone breaks down THC deposited on surfaces fr...,Second- and thirdhand tobacco smoke have recei...,Smoking emits reactive chemicals that remain i...,Ozone Holes,2020
3,"November 23, 2020",https://www.sciencedaily.com/releases/2020/11/...,Shift in atmospheric rivers could affect Antar...,Weather systems responsible for transporting m...,"Atmospheric rivers are long, narrow jets of ai...",Ozone Holes,2020
4,"September 24, 2020",https://www.sciencedaily.com/releases/2020/09/...,Climate pledges 'like tackling COVID-19 withou...,Current global pledges to tackle climate chang...,"In the Paris Agreement, nations agreed to limi...",Ozone Holes,2020


In [ ]:
# GenZ Paraphrase Generation

model = "gpt-5-nano"
api_key = "key"
client = OpenAI(api_key=api_key)

output_file = "SciDCC-paraphrased.jsonl"

MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds
MAX_WORKERS = 30

class StructuredOutput(BaseModel):
  paraphrased_text: str


def process_article(row):   
    article_body = row[4]
    article_title = row[2]
    category = row[5]

    paraphrase_instruct = f"""

    Paraphrase the article in a non-scientific, Gen Z style. Rewrite the entire article from beginning to end and do NOT skip or add sentences.
    Keep grammar and formatting consistent. Output only the rewritten article.
    If the article is too long to process in one go, output "Null". """

    retries = 0
    while retries < MAX_RETRIES:
        try:
            response = client.responses.parse(
                model=model,
                reasoning={"effort": "minimal"},
                input=[
                    {
                        "role": "system",
                        "content": paraphrase_instruct,
                                    },
                        {"role": "user",
                        "content": article_body}, 
                ],
                text_format=StructuredOutput,
            )

            body_variant = response.output_parsed.paraphrased_text.strip()

            bad_prefixes = (
                "i can", "i cannot", "sorry", "unable", "[", "definitely",
                "i need", "i am", "please", "i’m ready", "i’m", "i'm", "certainly", "here comes", "of course", "sure"
            )

            if body_variant.lower().startswith(bad_prefixes):
                return {
                "article_title:":article_title,
                "body_variant": None,
                "og_category":category
                }

            return {
                "article_title:":article_title,
                "body_variant":body_variant,
                "og_category":category
                }

        except Exception as e:
            retries += 1
            print(f"Error: {e} | Retrying {retries}/{MAX_RETRIES}")
            time.sleep(RETRY_DELAY + random.uniform(0,3))
    return {
            "article_title:":article_title,
            "body_variant":None,
            "og_category":category
            }

def main():
    rows = list(data.itertuples(index=False))
    total = len(rows)
    
    with open(output_file, "a", encoding="utf-8") as jsonfile:
        for start in range(0, total, MAX_WORKERS):
            end = min(start + MAX_WORKERS, total)
            batch_rows = rows[start:end]
            
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                batch_results = list(executor.map(process_article, batch_rows))
            for result in batch_results:
                if result:
                    jsonfile.write(json.dumps(result, ensure_ascii=False) +"\n")
                    jsonfile.flush() 
            print(f"Processed articles {start+1} to {end}...")

    print("Woohoo! All complete :)")

if __name__ == "__main__":
    main()


Processed articles 1 to 30...
Processed articles 31 to 60...
Processed articles 61 to 90...
Processed articles 91 to 120...
Error: 1 validation error for StructuredOutput
  Invalid JSON: EOF while parsing a string at line 1 column 842 [type=json_invalid, input_value='{"paraphrased_text":"So ...ls.” Rogers adds, “', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid | Retrying 1/3
Processed articles 121 to 150...
Processed articles 151 to 180...
Processed articles 181 to 210...
Processed articles 211 to 240...
Processed articles 241 to 270...
Processed articles 271 to 300...
Processed articles 301 to 330...
Processed articles 331 to 360...
Processed articles 361 to 390...
Processed articles 391 to 420...
Processed articles 421 to 450...
Processed articles 451 to 480...
Processed articles 481 to 510...
Processed articles 511 to 540...
Processed articles 541 to 570...
Processed articles 571 to 600...
Processed articles 601 to 630...
Processed